# Intermediate processing {#sec-img-intermediate-processing}



## Preamble

### Introduction

In this demo, we will continue processing the Xenium dataset from a human breast cancer biopsy section collected by @Janesick2023-high-res, which has already undergone quality control and minimal filtering; see @sec-img-quality-control.
The processing steps carried out here -- namely, normalization, dimension reduction, and clustering -- lay the foundating for a variety of downstream analysis tasks that will be covered in the next chapters.
Here, we will run standard non-spatial and spatially-aware approaches; for a more comprehensive overview of methodology and tools, see @sec-ind-dimensionality-reduction on dimension reduction and @sec-ind-clustering on clustering. 

### Dependencies

In [ ]:
# library(scran)
# library(scater)
# library(igraph)
library(Banksy)
library(scrapper)
library(ggplot2)
library(ggspavis)
library(patchwork)
library(OSTA.data)
library(SpatialExperiment)
# set seed for random number generation
# in order to make results reproducible
set.seed(20000229)
# load data from preceding 
# chapter (post quality control)
(spe <- readRDS("img-spe_qc.rds"))

Before getting started, we will retrieve the authors' cell type labels, 
which were obtained by transferring scFFPE-seq annotations (supervised); 
these comprise 20 subpopulations.

In [ ]:
# get annotations from 'BiocFileCache'
# (data has been retrieved already)
id <- "Xenium_HumanBreast1_Janesick"
pa <- OSTA.data_load(id, mol=FALSE)
dir.create(td <- tempfile())
unzip(pa, "annotation.csv", exdir=td)
df <- read.csv(list.files(td, full.names=TRUE))
# add annotations as cell metadata
cs <- match(spe$cell_id, df$Barcode)
spe$Label <- df$Annotation[cs]

## Normalization

Library size-based normalization, as typically used for scRNA-seq data, has 
been shown to be problematic for ST data, especially so for targeted panels 
underlying current commercial imaging-based ST platforms [@Atta2024-gene-count;
@Bhuva2024-library-size]. 
For lack of a better approach, we here use standard log-library size normalization. 
We caution readers, however, to keep an eye out in the literature for attempts 
to provide a better strategy. [See also @sec-ind-normalization.]{.aside}

In [ ]:
spe <- normalizeRnaCounts.se(spe)

## Feature selection

At this stage, we would typically perform selection of (e.g., highly variable) 
features; see @sec-seq-intermediate-processing. The dataset at hand, however, 
is *targeted* and relatively low-plex, so that 'interesting' features have 
already been selected by design (e.g., different targets will be included 
in immuno-oncology as opposed to neuroscience panels).

## Dimension Reduction

As a baseline, we will perform principal component analysis (PCA), which 
underlies many standard scRNA-seq analysis pipelines, such as (spatially 
unaware) graph-based clustering based on a shared nearest neighbor (SNN) 
graph and the Leiden or Louvain algorithm for community detection.

In [ ]:
spe <- runPca.se(spe, features=rownames(spe), number=20)

For comparison, we will also perform spatially-aware dimension reduction 
with `r BiocStyle::Biocpkg("BANKSY")` [@Singhal2024-BANKSY]; 
see @sec-ind-dimensionality-reduction.

In [ ]:
spe <- computeBanksy(spe, assay_name="logcounts")
spe <- runBanksyPCA(spe, npcs=20, lambda=0.2)

To not confuse different types of PCs, we rename `reducedDims` to end in 
`_sp` and `_tx` for spatially aware and unaware results, respectively.

In [ ]:
reducedDimNames(spe) <- c("PCA_tx", "PCA_sp")

## Clustering

Here, we perform standard graph-based clustering by (i) constructing a shared 
nearest neighbor (SNN) graph and (ii) using the Leiden algorithm for community 
detection. By basing the SNN graph on standard and `BANKSY` PCs, respectively, 
we can obtain non-spatial as well as spatially aware assignments:

In [ ]:
# PCA-based shared nearest-neighbor (SNN) graph;
# cluster via Leiden community detection algorithm
pcs <- c(Leiden="PCA_tx", Banksy="PCA_sp")
for (k in names(pcs)) {
    spe <- clusterGraph.se(spe, 
        method="leiden", resolution=0.7, 
        output.name=k, reddim.type=pcs[[k]], 
        more.build.args=list(weight.scheme="jaccard"))
}

## Visualization

Let's visualize the assignment obtains from non-spatial and spatially aware clustering:

In [ ]:
#| code-fold: true
spe$in_tissue <- 1; spe$x_centroid <- spe$y_centroid <- NULL
lapply(c("Label", "Leiden", "Banksy"), \(.) {
    plotCoords(spe, annotate=., point_size = 0.1)
}) |>
    wrap_plots(nrow=1) &
    theme(legend.key.size=unit(0, "lines")) &
    scale_color_manual(values=unname(pals::trubetskoy()))

## Appendix

### Save data {.unnumbered}

In [ ]:
colLabels(spe) <- spe$Banksy
saveRDS(spe, "img-spe_cl.rds")

### References {.unnumbered}